In [82]:
import delta

#import
import ConnectionConfig as cc
from delta.tables import DeltaTable
from datetime import datetime
debugging_mode=True


In [83]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("dimUserIncrementalLoad",4)
spark.getActiveSession()

run_timestamp = datetime.now()

Environment variables are set...


In [84]:
#EXTRACT

#dt ophalen en naar view omvoren
dt_userDim = delta.DeltaTable.forPath(spark,"./delta/USER_DIM")
dt_userDim.toDF().createOrReplaceTempView("dimuser_current")

#debug code
spark.sql("select *  from dimuser_current ").show()

+--------------------+----------+-----------+--------------------+--------------------+---------------+---------+-------+--------------------+-------+-------+--------------------+
|              userId|first_Name|  last_Name|               email|             address|experienceLevel|dedicator|country|           scd_start|scd_end|current|                 md5|
+--------------------+----------+-----------+--------------------+--------------------+---------------+---------+-------+--------------------+-------+-------+--------------------+
|[00 02 66 4B E2 A...|    Mellie|       Mraz|Mellie.Mraz@russe...|   Freeda Canyon 503|        Amateur|    false|     IN|2025-10-12 13:46:...|   NULL|   true|78b07a20e6d1bb02d...|
|[00 05 0C 95 84 B...|   Alfreda|     Daniel|Alfreda.Daniel@wy...|     Edmund Pike 610|        Amateur|    false|     IN|2025-10-12 13:46:...|   NULL|   true|78b07a20e6d1bb02d...|
|[00 07 66 C4 1B 4...|  Alexanne|     Cronin|Alexanne.Cronin@c...|     Klocko Cape 822|         Pira

In [85]:
#set connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [ ]:
#EXTRACT

#get info, complete user terug samen stellen
# user tabel
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "user_table") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("users")

# treusure log tabel
df_treasure_log = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_log.createOrReplaceTempView("treasure_log")

#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

#city tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "city") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("city")

#country tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "country") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("country")

In [ ]:
#TRANSFORM

#base user
base_user = spark.sql("""
                      SELECT u.id                            AS userId,
                             u.first_name                    AS first_Name,
                             u.last_name                     AS last_Name,
                             u.mail                          AS email,
                             u.city_city_id                  as cityId,
                             CONCAT(u.street, ' ', u.number) AS address,
                             COALESCE(tl.treasure_count, 0)  AS treasure_count
                      FROM users u
                               LEFT JOIN (SELECT hunter_id, COUNT(*) AS treasure_count
                                          FROM treasure_log
                                          GROUP BY hunter_id) tl ON u.id = tl.hunter_id
                      """)

base_user.createOrReplaceTempView("baseUser")

In [ ]:
#TRANSFORM

#expierenceLevel berekenen
expierencelevel_user = spark.sql("""
                                 SELECT userId,
                                        CASE
                                            WHEN treasure_count IS NULL OR treasure_count = 0 THEN 'Starter'
                                            WHEN treasure_count < 4 THEN 'Amateur'
                                            WHEN treasure_count BETWEEN 4 AND 10 THEN 'Professional'
                                            ELSE 'Pirate'
                                            END AS experienceLevel
                                 FROM baseUser
                                 """)
expierencelevel_user.createOrReplaceTempView("expierenceLevelUser")

In [ ]:
#TRANSFORM

#dedicator berekenen
dedicator_user = spark.sql("""
                           SELECT u.userId,
                                  CASE WHEN COUNT(t.id) > 0 THEN TRUE ELSE FALSE END AS dedicator
                           FROM baseUser u
                                    LEFT JOIN treasure t ON u.userId = t.owner_id
                           GROUP BY u.userId
                           """)
dedicator_user.createOrReplaceTempView("dedicatorUser")

In [ ]:
#TRANSFORM

#Country berekenen
country_user = spark.sql("""
                         SELECT u.userId,
                                co.code as country
                         FROM baseUser u
                                  JOIN city ct on ct.city_id = u.cityId
                                  join country co on co.code = ct.country_code
                         """)

country_user.createOrReplaceTempView("countryUser")

In [86]:
#TRANSFORM

#zet alles samen
complete_user = spark.sql(f"""
SELECT
    b.userId as source_user_id,
    b.first_Name as source_first_name,
    b.last_Name as source_last_name,
    b.email as source_email,
    b.address as source_address,
    e.experienceLevel as source_experienceLevel,
    d.dedicator as source_dedicator,
    c.country as source_country,
    TRUE AS source_current,
    md5(CONCAT(e.experienceLevel,d.dedicator,b.first_Name,b.last_Name)) AS source_md5
FROM baseUser b
LEFT JOIN expierenceLevelUser e ON b.userId = e.userId
LEFT JOIN dedicatorUser d ON b.userId = d.userId
LEFT JOIN countryUser c ON b.userId = c.userId
""")

complete_user.createOrReplaceTempView("completeUser_new")
#spark.sql("select * from completeUser_new").show()

In [87]:
#TRANSFORM

#detecteer en opslagen in tempview veranderingen
detectedChanges = spark.sql(f"select * \
                          from completeUser_new source \
                          left outer join dimuser_current dwh on dwh.userId == source.source_user_id and dwh.current == true \
                          where dwh.userId is null or dwh.md5 <> source.source_md5")

detectedChanges.createOrReplaceTempView("detectedChanges")

#detectedChanges.show()


In [88]:
#TRANSFORM

#upsert
df_upsert = spark.sql(f"""
SELECT
    source_user_id as userId,
    source_first_name as first_Name,
    source_last_name as last_Name,
    source_email as email,
    source_address as address,
    source_experienceLevel as experienceLevel,
    source_dedicator as dedicator,
    source_country as country,
    to_timestamp('{run_timestamp}') as scd_start,
    to_timestamp(null) AS scd_end,
    source_md5 AS md5,
    TRUE AS current
    FROM detectedChanges
union
SELECT
    userId,
    first_Name,
    last_Name,
    email,
    address ,
    experienceLevel ,
    dedicator ,
    country,
    scd_start,
    to_timestamp('{run_timestamp}') as scd_end,
    md5,
    false
    FROM detectedChanges
    where current is not null
                          """)

df_upsert.createOrReplaceTempView("upserts")

#debug code
#spark.sql("select * from upserts").show()

In [89]:
#TRANSFORM

#merge van alles
spark.sql("""
MERGE INTO dimUser_current AS target
USING upserts AS source
ON target.userId = source.userId
  AND source.current = false
  AND target.current = true
WHEN MATCHED THEN
  UPDATE SET
    target.scd_end = source.scd_end,
    target.current = source.current
WHEN NOT MATCHED THEN
  INSERT (
    userId,
    first_Name,
    last_Name,
    email,
    address,
    experienceLevel,
    dedicator,
    country,
    scd_start,
    scd_end,
    md5,
    current
  )
  VALUES (
    source.userId,
    source.first_Name,
    source.last_Name,
    source.email,
    source.address,
    source.experienceLevel,
    source.dedicator,
    source.country,
    source.scd_start,
    source.scd_end,
    source.md5,
    source.current
  )
""")

dt_userDim.toDF().sort("userId", "scd_start").show(50)

+--------------------+----------+-----------+--------------------+--------------------+---------------+---------+-------+--------------------+-------+-------+--------------------+
|              userId|first_Name|  last_Name|               email|             address|experienceLevel|dedicator|country|           scd_start|scd_end|current|                 md5|
+--------------------+----------+-----------+--------------------+--------------------+---------------+---------+-------+--------------------+-------+-------+--------------------+
|[00 00 1C BF A8 E...|     Yanis|      Andre|Yanis.Andre@girar...|Allée, Voie du Ba...|        Amateur|    false|     FR|2025-10-12 13:46:...|   NULL|   true|78b07a20e6d1bb02d...|
|[00 00 2B 0B 08 D...|    Jaeden|Cruickshank|Jaeden.Cruickshan...|     Huels Green 233|        Amateur|    false|     US|2025-10-12 13:46:...|   NULL|   true|78b07a20e6d1bb02d...|
|[00 00 2B 76 18 6...|     Sonny|    Wiegand|Sonny.Wiegand@spi...|     Isabel View 112|        Amate

In [90]:
#stop spark sessie
spark.stop()